# 2606-Data Ecosystems and Governance in Organizations

# 0. Load Data

In [100]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

with open('data/raw_credit_applications.json') as f:
    data = json.load(f)

print(f"Total records in raw file: {len(data)}")

Total records in raw file: 502


In [ ]:
# Connect to MongoDB and upsert records
client = MongoClient()
db = client['novacred']
collection = db['applications']
collection.drop()

for record in data:
    collection.replace_one({'_id': record['_id']}, record, upsert=True)

print(f"Records in MongoDB after upsert: {collection.count_documents({})}")
print("(Duplicate IDs were overwritten with their latest version)")

Records in MongoDB after upsert: 500
(Duplicate IDs were overwritten with their latest version)


# 1. Duplicates

In [102]:
# Find duplicates before inserting
ids = [record['_id'] for record in data]
id_counts = Counter(ids)
duplicate_ids = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Total records in file : {len(data)}")
print(f"Unique _id values     : {len(set(ids))}")
print(f"Duplicate IDs         : {list(duplicate_ids.keys())}")

# Show duplicates
for dup_id in duplicate_ids:
    versions = [r for r in data if r['_id'] == dup_id]
    print(f"\n--- Versions of {dup_id} ---")
    for i, v in enumerate(versions):
        print(f"  Version {i+1}: notes='{v.get('notes', 'none')}', keys={sorted(v.keys())}")

Total records in file : 502
Unique _id values     : 500
Duplicate IDs         : ['app_042', 'app_001']

--- Versions of app_042 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='RESUBMISSION', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

--- Versions of app_001 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='DUPLICATE_ENTRY_ERROR', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']


**Notes**:

- The raw dataset contained 502 records, of which 500 were unique applications. app_042 was flagged as a resubmission, and app_001 was flagged as a duplicate entry error. For app_042, we'll keep the resubmission as the most recent version to honor the applicant's latest intent.The duplicate entry error of app_001 will be resolved by restoring the original record.

In [103]:
# SSNs as unique IDs. (different IDs, same person?)
ssns = [(r['_id'], r['applicant_info'].get('ssn')) for r in data]
ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

print(f"Duplicate SSNs (same SSN across different app IDs): {len(dup_ssns)}")
if dup_ssns:
    for ssn, count in dup_ssns.items():
        app_ids = [id_ for id_, s in ssns if s == ssn]
        print(f"  SSN {ssn} appears in: {app_ids}")

Duplicate SSNs (same SSN across different app IDs): 3
  SSN 652-70-5530 appears in: ['app_042', 'app_042']
  SSN 937-72-8731 appears in: ['app_101', 'app_234']
  SSN 780-24-9300 appears in: ['app_088', 'app_016']


**Notes**:
- SSN 652-70-5530: Both entries are app_042, so this is just your already-known duplicate ID showing up twice in the raw data. Not a new issue.
- SSN 937-72-8731: Appears in app_101 AND app_234, two different application IDs. Same person, two applications.
- SSN 780-24-9300: Appears in app_088 AND app_016, same situation.

## 1.1 Handle Duplicates

In [104]:
# Fix duplicte entry of app_001, keep correct version
# app_001 — restore original, discard DUPLICATE_ENTRY_ERROR version
original_app_001 = next(r for r in data 
                        if r['_id'] == 'app_001' 
                        and 'DUPLICATE_ENTRY_ERROR' not in str(r.get('notes', '')))

# Change to correct version
collection.replace_one({'_id': 'app_001'}, original_app_001, upsert=True)

print(f"\nFinal document count in MongoDB: {collection.count_documents({})}")


Final document count in MongoDB: 500


In [105]:
# Fix SSN duplicates
unique_data = {r['_id']: r for r in data}
ssns = [(id_, r['applicant_info'].get('ssn')) 
        for id_, r in unique_data.items()]

ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

for ssn, count in dup_ssns.items():
    app_ids = [id_ for id_, s in ssns if s == ssn]
    print(f"  SSN {ssn} appears in: {app_ids}")

# Flag in MongoDB for human review
for ssn in dup_ssns:
    app_ids = [id_ for id_, s in ssns if s == ssn]
    for app_id in app_ids:
        collection.update_one(
            {'_id': app_id},
            {'$set': {'data_quality_flag': 'duplicate_ssn'}}
        )
        print(f"  Flagged {app_id} with 'duplicate_ssn'")

  SSN 937-72-8731 appears in: ['app_101', 'app_234']
  SSN 780-24-9300 appears in: ['app_088', 'app_016']
  Flagged app_101 with 'duplicate_ssn'
  Flagged app_234 with 'duplicate_ssn'
  Flagged app_088 with 'duplicate_ssn'
  Flagged app_016 with 'duplicate_ssn'


The reason you flag rather than delete is important for your README — these are two different application IDs for the same person, so automatically removing one could discard a legitimate credit decision. A human needs to review whether it's a reapplication, fraud, or a data entry error.

# 2. Data Type Check

In [106]:
# Check data types in raw json
fields_to_check = {
    # Financials (numeric)
    'financials.annual_income'        : ('financials', 'annual_income',         (int, float)),
    'financials.credit_history_months': ('financials', 'credit_history_months', (int,)),
    'financials.debt_to_income'       : ('financials', 'debt_to_income',        (int, float)),
    'financials.savings_balance'      : ('financials', 'savings_balance',       (int, float)),
    # Applicant info (all strings)
    'applicant_info.full_name'        : ('applicant_info', 'full_name',         (str,)),
    'applicant_info.email'            : ('applicant_info', 'email',             (str,)),
    'applicant_info.ssn'              : ('applicant_info', 'ssn',               (str,)),
    'applicant_info.ip_address'       : ('applicant_info', 'ip_address',        (str,)),
    'applicant_info.gender'           : ('applicant_info', 'gender',            (str,)),
    'applicant_info.date_of_birth'    : ('applicant_info', 'date_of_birth',     (str,)),
    'applicant_info.zip_code'         : ('applicant_info', 'zip_code',          (str,)),
    # Decision
    'decision.loan_approved'          : ('decision', 'loan_approved',           (bool,)),
    'decision.interest_rate'          : ('decision', 'interest_rate',           (int, float)),
    'decision.approved_amount'        : ('decision', 'approved_amount',         (int, float)),
    'decision.rejection_reason'       : ('decision', 'rejection_reason',        (str,)),
}

print(f"{'Field':<40} {'Data Types Found':<30}")
print("-" * 70)
for label, (section, field, expected_types) in fields_to_check.items():
    type_counts = Counter(
        type(r.get(section, {}).get(field)).__name__
        for r in data
        if r.get(section, {}).get(field) is not None
    )
    print(f"{label:<40} {str(dict(type_counts)):<30}")

# Check spending_behavior separately
spending_amount_issues = [
    r['_id'] for r in data
    for item in r.get('spending_behavior', [])
    if not isinstance(item.get('amount'), (int, float))
]

spending_category_issues = [
    r['_id'] for r in data
    for item in r.get('spending_behavior', [])
    if not isinstance(item.get('category'), str)
]

print(f"spending_behavior.amount: {len(spending_amount_issues)} non-numeric records")
print(f"spending_behavior.category: {len(spending_category_issues)} non-string records")

Field                                    Data Types Found              
----------------------------------------------------------------------
financials.annual_income                 {'int': 488, 'str': 8, 'float': 1}
financials.credit_history_months         {'int': 502}                  
financials.debt_to_income                {'float': 502}                
financials.savings_balance               {'int': 502}                  
applicant_info.full_name                 {'str': 502}                  
applicant_info.email                     {'str': 502}                  
applicant_info.ssn                       {'str': 497}                  
applicant_info.ip_address                {'str': 497}                  
applicant_info.gender                    {'str': 501}                  
applicant_info.date_of_birth             {'str': 501}                  
applicant_info.zip_code                  {'str': 501}                  
decision.loan_approved                   {'bool': 502}       

In [107]:
# Fix annual_income type issues in MongoDB
# Identify records where annual_income is a string instead of a number
type_issues = [
    (r['_id'], r['financials'].get('annual_income'))
    for r in data
    if isinstance(r.get('financials', {}).get('annual_income'), str)
]

# Fix in MongoDB
for app_id, val in type_issues:
    cleaned = int(float(str(val).replace(',', '').strip()))
    collection.update_one(
        {'_id': app_id},
        {'$set': {'financials.annual_income': int(float(str(val)))}}
    )